# SENN on FashionMNIST — Exploration Report

This notebook explores the results of two SENN runs trained on **FashionMNIST** (λ = 1e-4):
- **c10**: 10 concepts (`fashion_mnist_lambda1e-4_c10_seed29`)
- **c5**: 5 concepts (`fashion_mnist_lambda1e-4_c5_seed29`)

For each run we analyse:
1. Training and validation curves
2. Test accuracy
3. SENN explanations on random samples
4. SENN explanations for one sample per class
5. Concept prototypes (highest activation + highest contrast)

A final section compares the two runs side-by-side and provides a template for a future λ-sweep comparison.

**FashionMNIST class labels:**

| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 |
|---|---|---|---|---|---|---|---|---|---|
| T-shirt/top | Trouser | Pullover | Dress | Coat | Sandal | Shirt | Sneaker | Bag | Ankle boot |

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from types import SimpleNamespace
from pathlib import Path

# SENN modules
from senn.trainer import SENN_Trainer
from senn.utils.plot_utils import show_explainations, get_comparison_plot
from senn.utils.concept_representations import highest_activations, highest_contrast

plt.style.use('seaborn-v0_8-talk')
%matplotlib inline

FASHION_MNIST_CLASSES = [
    'T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
    'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot'
]

print('Imports OK')

In [ ]:
# ─── Helper functions ────────────────────────────────────────────────────────

def load_senn(config_path, device='cpu'):
    """Load a trained SENN from config + best_model.pt checkpoint."""
    with open(config_path, 'r') as f:
        config = json.load(f)
    config['device'] = device
    config['train'] = False
    config['load_checkpoint'] = 'best_model.pt'
    config = SimpleNamespace(**config)
    t = SENN_Trainer(config)
    t.model.eval()
    return t


def plot_training_curves(exp_name, results_dir='results'):
    """Plot accuracy and loss curves from the saved CSVs."""
    train_csv = Path(results_dir) / exp_name / 'accuracies_losses_train.csv'
    valid_csv = Path(results_dir) / exp_name / 'accuracies_losses_valid.csv'

    df_train = pd.read_csv(train_csv)
    df_valid = pd.read_csv(valid_csv)

    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    fig.suptitle(f'Training Curves — {exp_name}', fontsize=13, fontweight='bold')

    # --- Accuracy ---
    axes[0].plot(df_train['Step'], df_train['Accuracy'], label='Train', alpha=0.7)
    axes[0].plot(df_valid['Step'], df_valid['Accuracy'], label='Validation',
                 marker='o', linewidth=2)
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Step')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)

    # --- Total Loss ---
    axes[1].plot(df_train['Step'], df_train['Loss'], label='Train', alpha=0.7)
    axes[1].plot(df_valid['Step'], df_valid['Loss'], label='Validation',
                 marker='o', linewidth=2)
    axes[1].set_title('Total Loss')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)

    # --- Loss breakdown (train only) ---
    axes[2].plot(df_train['Step'], df_train['Classification_Loss'],
                 label='Classification', alpha=0.8)
    axes[2].plot(df_train['Step'], df_train['Concept_Loss'],
                 label='Concept', alpha=0.8)
    axes[2].plot(df_train['Step'], df_train['Robustness_Loss'],
                 label='Robustness', alpha=0.8)
    axes[2].set_title('Loss Components (Train)')
    axes[2].set_xlabel('Step')
    axes[2].set_ylabel('Loss')
    axes[2].legend()
    axes[2].grid(True)

    plt.tight_layout()
    plt.show()


def get_one_per_class(test_loader, num_classes=10, device='cpu'):
    """Return a stacked tensor with one image per class."""
    examples = {}
    for x, y in test_loader:
        for i in range(len(y)):
            label = y[i].item()
            if label not in examples:
                examples[label] = x[i]
            if len(examples) == num_classes:
                break
        if len(examples) == num_classes:
            break
    return torch.stack([examples[i] for i in range(num_classes)]).to(device)


def show_per_class_explanations(model, test_loader, class_names,
                                device='cpu', group_size=5):
    """Show SENN explanations (image + theta + h) for one sample per class."""
    images = get_one_per_class(test_loader, num_classes=len(class_names), device=device)
    model.eval()
    with torch.no_grad():
        y_pred, (concepts, relevances), _ = model(images)
    y_pred_idx = y_pred.argmax(1)

    num_concepts = concepts.shape[1]
    concept_lim = max(abs(concepts.min().item()), abs(concepts.max().item())) + 0.1
    y_labels = [f'C{j+1}' for j in range(num_concepts - 1, -1, -1)]

    for group_start in range(0, len(class_names), group_size):
        group_end = min(group_start + group_size, len(class_names))
        n = group_end - group_start

        fig, axes = plt.subplots(3, n, figsize=(3.2 * n, 3.2 * 3))
        if n == 1:
            axes = axes[:, np.newaxis]

        for col, idx in enumerate(range(group_start, group_end)):
            true_label = class_names[idx]
            pred_label = class_names[y_pred_idx[idx].item()]
            title_color = 'green' if true_label == pred_label else 'red'

            # Row 0 — input image
            axes[0, col].imshow(images[idx].squeeze().cpu(), cmap='gray')
            axes[0, col].set_title(
                f'True: {true_label}\nPred: {pred_label}',
                fontsize=9, color=title_color
            )
            axes[0, col].axis('off')

            # Row 1 — relevance scores theta
            rs = relevances[idx, :, y_pred_idx[idx].item()].cpu()
            colors_r = ['b' if v > 0 else 'r' for v in rs.tolist()]
            colors_r.reverse()
            axes[1, col].barh(np.arange(num_concepts), np.flip(rs.numpy()),
                              color=colors_r, align='center')
            axes[1, col].set_yticks(np.arange(num_concepts))
            axes[1, col].set_yticklabels(y_labels, fontsize=8)
            axes[1, col].set_xlim(-1.1, 1.1)
            axes[1, col].axvline(0, color='black', linewidth=0.6, linestyle='--')
            if col == 0:
                axes[1, col].set_ylabel('Relevances θ', fontsize=10)

            # Row 2 — concept activations h
            cs = concepts[idx].flatten().cpu()
            colors_c = ['b' if v > 0 else 'r' for v in cs.tolist()]
            colors_c.reverse()
            axes[2, col].barh(np.arange(num_concepts), np.flip(cs.numpy()),
                              color=colors_c, align='center')
            axes[2, col].set_yticks(np.arange(num_concepts))
            axes[2, col].set_yticklabels(y_labels, fontsize=8)
            axes[2, col].set_xlim(-concept_lim, concept_lim)
            axes[2, col].axvline(0, color='black', linewidth=0.6, linestyle='--')
            if col == 0:
                axes[2, col].set_ylabel('Concepts h', fontsize=10)

        plt.suptitle(
            f'Classes {group_start} – {group_end - 1}: '
            + ', '.join(class_names[group_start:group_end]),
            fontsize=11, fontweight='bold'
        )
        plt.tight_layout()
        plt.show()


print('Helper functions loaded.')

---
## 1. Run c10 — 10 Concepts, λ = 1e-4
`fashion_mnist_lambda1e-4_c10_seed29`

In [ ]:
CONFIG_C10 = 'configs/fashion_mnist_lambda1e-4_c10_seed29.json'
EXP_C10    = 'fashion_mnist_lambda1e-4_c10_seed29'

print(f'Loading model: {EXP_C10} ...')
trainer_c10  = load_senn(CONFIG_C10, device='cpu')
model_c10    = trainer_c10.model
test_loader_c10 = trainer_c10.test_loader
print('Model loaded successfully.')

### 1.1 Training and Validation Curves

In [ ]:
plot_training_curves(EXP_C10)

### 1.2 Test Accuracy

In [ ]:
acc_c10 = trainer_c10.test()
print(f'\n>>> Test Accuracy (c10): {acc_c10:.4f}  ({acc_c10 * 100:.2f} %)')

### 1.3 SENN Explanations — Random Samples

Each panel shows: **input image** · **relevance scores θ** (how much each concept contributes to the prediction) · **concept activations h** (the values of the latent concepts).

In [ ]:
show_explainations(
    model_c10, test_loader_c10,
    dataset='fashion_mnist',
    num_explanations=6,
    batch_size=200
)

### 1.4 SENN Explanations — One Sample per Class

The title is **green** when the model predicts the correct class, **red** otherwise.

In [ ]:
show_per_class_explanations(
    model_c10, test_loader_c10,
    class_names=FASHION_MNIST_CLASSES,
    device='cpu',
    group_size=5
)

### 1.5 Concept Prototypes — Highest Activation

For each concept, the **9 test-set images that produce the highest activation** for that concept.

In [ ]:
highest_activations(
    model_c10, test_loader_c10,
    num_concepts=10,
    num_prototypes=9
)

### 1.6 Concept Prototypes — Highest Contrast

For each concept, the **9 images that activate it most exclusively** (high activation for that concept, low for all others).

In [ ]:
highest_contrast(
    model_c10, test_loader_c10,
    num_concepts=10,
    num_prototypes=9
)

---
## 2. Run c5 — 5 Concepts, λ = 1e-4
`fashion_mnist_lambda1e-4_c5_seed29`

In [ ]:
CONFIG_C5 = 'configs/fashion_mnist_lambda1e-4_c5_seed29.json'
EXP_C5    = 'fashion_mnist_lambda1e-4_c5_seed29'

print(f'Loading model: {EXP_C5} ...')
trainer_c5  = load_senn(CONFIG_C5, device='cpu')
model_c5    = trainer_c5.model
test_loader_c5 = trainer_c5.test_loader
print('Model loaded successfully.')

### 2.1 Training and Validation Curves

In [ ]:
plot_training_curves(EXP_C5)

### 2.2 Test Accuracy

In [ ]:
acc_c5 = trainer_c5.test()
print(f'\n>>> Test Accuracy (c5): {acc_c5:.4f}  ({acc_c5 * 100:.2f} %)')

### 2.3 SENN Explanations — Random Samples

In [ ]:
show_explainations(
    model_c5, test_loader_c5,
    dataset='fashion_mnist',
    num_explanations=6,
    batch_size=200
)

### 2.4 SENN Explanations — One Sample per Class

In [ ]:
show_per_class_explanations(
    model_c5, test_loader_c5,
    class_names=FASHION_MNIST_CLASSES,
    device='cpu',
    group_size=5
)

### 2.5 Concept Prototypes — Highest Activation

In [ ]:
highest_activations(
    model_c5, test_loader_c5,
    num_concepts=5,
    num_prototypes=9
)

### 2.6 Concept Prototypes — Highest Contrast

In [ ]:
highest_contrast(
    model_c5, test_loader_c5,
    num_concepts=5,
    num_prototypes=9
)

---
## 3. Comparison: c5 vs c10  (λ = 1e-4)

Side-by-side comparison of the two runs.

In [ ]:
# ─── Accuracy table ──────────────────────────────────────────────────────────
results = [
    ('c10  (10 concepts, λ=1e-4)', acc_c10, EXP_C10),
    ('c5   ( 5 concepts, λ=1e-4)', acc_c5,  EXP_C5),
]

header = f"{'Model':<38} {'Test Acc':>12}  {'Best Valid Acc':>15}"
print(header)
print('─' * 68)
for label, test_acc, exp in results:
    valid_acc = pd.read_csv(
        Path('results') / exp / 'accuracies_losses_valid.csv'
    )['Accuracy'].max()
    print(f'{label:<38} {test_acc:>11.4f}  {valid_acc:>14.4f}')
print('─' * 68)

In [ ]:
# ─── Validation curves side by side ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Validation Curves: c10 vs c5  (λ = 1e-4)', fontsize=13, fontweight='bold')

palette = {'c10 (10 concepts)': 'tab:blue', 'c5  (5 concepts)': 'tab:orange'}
exps    = [(EXP_C10, 'c10 (10 concepts)'), (EXP_C5, 'c5  (5 concepts)')]

for exp_name, label in exps:
    df_v = pd.read_csv(Path('results') / exp_name / 'accuracies_losses_valid.csv')
    color = palette[label]
    axes[0].plot(df_v['Step'], df_v['Accuracy'], label=label, color=color,
                 marker='o', linewidth=2)
    axes[1].plot(df_v['Step'], df_v['Loss'],     label=label, color=color,
                 marker='o', linewidth=2)

for ax, title, ylabel in zip(axes,
                              ['Validation Accuracy', 'Validation Total Loss'],
                              ['Accuracy', 'Loss']):
    ax.set_title(title)
    ax.set_xlabel('Step')
    ax.set_ylabel(ylabel)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Training loss component comparison ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training Loss Components: c10 vs c5  (λ = 1e-4)', fontsize=13, fontweight='bold')

components = ['Classification_Loss', 'Concept_Loss', 'Robustness_Loss']
titles     = ['Classification Loss', 'Concept Loss', 'Robustness Loss']

for exp_name, label, color in [
    (EXP_C10, 'c10 (10 concepts)', 'tab:blue'),
    (EXP_C5,  'c5  (5 concepts)',  'tab:orange'),
]:
    df_t = pd.read_csv(Path('results') / exp_name / 'accuracies_losses_train.csv')
    for ax, comp, title in zip(axes, components, titles):
        ax.plot(df_t['Step'], df_t[comp], label=label, color=color, alpha=0.8)
        ax.set_title(title)
        ax.set_xlabel('Step')
        ax.legend()
        ax.grid(True)

plt.tight_layout()
plt.show()

---
## 4. Template: λ Regularization Sweep  *(to be completed)*

When additional runs with different values of the robustness regularization strength λ are available,
fill in the config list below and run the cell.

In [ ]:
# ─── λ comparison — fill in config filenames as more runs become available ───
#
# from senn.utils.plot_utils import plot_lambda_accuracy
#
# config_list = [
#     'fashion_mnist_lambda???_c5_seed29.json',   # λ = ???
#     'fashion_mnist_lambda1e-4_c5_seed29.json',  # λ = 1e-4  (already done)
#     'fashion_mnist_lambda1e-1_seed29.json',     # λ = 1e-1  (already done)
# ]
#
# plot_lambda_accuracy(config_list, valid=True)

print('λ comparison template ready — uncomment and run once more configs are available.')